# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect to the warehouse and rebuild the March feature frame + Week-4 baseline

In [22]:
%pip install -q duckdb scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SNAPSHOT = "2026-03-31"   
MONTH = "2026-03"          
PREV_MONTH = "2026-02"     

print("Connected to Hugging Face warehouse")
print(f"Snapshot date    : {SNAPSHOT}")
print(f"Feature month    : {MONTH}")
print(f"Comparison month : {PREV_MONTH}")

Connected to Hugging Face warehouse
Snapshot date    : 2026-03-31
Feature month    : 2026-03
Comparison month : 2026-02


### 0.1 Rebuild the March / February aggregates and content metadata

In [24]:
# March aggregate -- this month's observable signals, per content item.
con.execute(f"""
    CREATE OR REPLACE TABLE march_agg AS
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)  AS gsc_impressions_mar,
        SUM(gsc_clicks)       AS gsc_clicks_mar,
        AVG(gsc_avg_position) AS avg_position_mar,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_pageviews ELSE NULL END)        AS ga4_pageviews_mar,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE NULL END) AS ga4_engaged_sessions_mar
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""")

# February aggregate -- comparison window, used ONLY to build the proxy label (never a feature).
con.execute(f"""
    CREATE OR REPLACE TABLE feb_agg AS
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_feb
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={PREV_MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""")

# Content metadata -- static fields, one row per content item. Same columns w04 used (for the baseline
# rebuild) plus word_count/backlinks/content_type, which the model gets to use as real features.
con.execute(f"""
    CREATE OR REPLACE TABLE content_meta AS
    SELECT content_hash_id, client_hash_id, content_type, word_count, backlinks,
           content_created_date, content_updated_date,
           last_optimized_date, optimization_eligible_date
    FROM read_parquet('{BASE}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""")

print('march_agg, feb_agg, content_meta built.')
con.sql('SELECT COUNT(*) AS march_rows FROM march_agg').show()
con.sql('SELECT COUNT(*) AS content_rows FROM content_meta').show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

march_agg, feb_agg, content_meta built.
┌────────────┐
│ march_rows │
│   int64    │
├────────────┤
│     176738 │
└────────────┘

┌──────────────┐
│ content_rows │
│    int64     │
├──────────────┤
│       411540 │
└──────────────┘



In [25]:
# Join into one content-item-per-row frame with the proxy label -- identical definition to ML-04/ML-07.
feature_frame = con.sql(f"""
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.gsc_impressions_mar,
        m.gsc_clicks_mar,
        m.avg_position_mar,
        m.ga4_pageviews_mar,
        m.ga4_engaged_sessions_mar,
        c.content_created_date,
        c.content_updated_date,
        c.last_optimized_date,
        c.optimization_eligible_date,
        c.word_count,
        c.content_type,
        c.backlinks,
        f.gsc_impressions_feb,
        -- proxy label: impressions dropped >20% March vs February. Context/target only -- never a feature.
        CASE
            WHEN f.gsc_impressions_feb > 0
             AND (m.gsc_impressions_mar - f.gsc_impressions_feb) * 1.0 / f.gsc_impressions_feb <= -0.20
            THEN 1 ELSE 0
        END AS is_declining_proxy
    FROM march_agg m
    JOIN feb_agg f USING (client_hash_id, content_hash_id)
    JOIN content_meta c USING (client_hash_id, content_hash_id)
    WHERE f.gsc_impressions_feb > 0
""").df()

# Derived, March-only quantities -- all knowable at the SNAPSHOT decision moment (w03 Section 3.5).
feature_frame['ctr_mar'] = (
    feature_frame['gsc_clicks_mar'] / feature_frame['gsc_impressions_mar'].replace(0, np.nan)
)
feature_frame['days_since_update'] = (
    pd.Timestamp(SNAPSHOT) - pd.to_datetime(feature_frame['content_updated_date'])
).dt.days.clip(lower=0)  # floor at 0 -- a future update date is not a real staleness signal
feature_frame['content_age_days'] = (
    pd.Timestamp(SNAPSHOT) - pd.to_datetime(feature_frame['content_created_date'])
).dt.days

print(f'Feature frame rows: {len(feature_frame)}')
feature_frame[['client_hash_id','content_hash_id','gsc_impressions_mar','ctr_mar',
               'avg_position_mar','days_since_update','is_declining_proxy']].head()

Feature frame rows: 134086


,client_hash_id,content_hash_id,gsc_impressions_mar,ctr_mar,avg_position_mar,days_since_update,is_declining_proxy
0,client_157ffe4d4a595515,content_88b1daa0918ed139,276.0,0.007246,5.428830,0,0
1,client_157ffe4d4a595515,content_88b4c2b2050326d0,1277.0,0.000783,2.970158,0,0
2,client_157ffe4d4a595515,content_88c03e8eb0d7089b,726.0,0.002755,8.169038,0,0
3,client_157ffe4d4a595515,content_88fa1a42b0ed7612,168.0,0.005952,9.234822,0,0
4,client_157ffe4d4a595515,content_88fe52bf9d05da18,85.0,0.000000,8.027095,0,0


### 0.2 Rebuild the Week-4 baseline score on this exact frame

Same rule, same thresholds, same peer-group CTR comparison as `w04_baseline_score.ipynb` — copied verbatim (not re-derived) so the comparison in Section 3 is guaranteed apples-to-apples: same rows, same label, same score definition my Week-5 model has to beat.

In [26]:
feature_frame['is_eligible'] = (
    (feature_frame['gsc_impressions_mar'] >= 500)
    & (feature_frame['content_age_days'] >= 90)
    & (
        feature_frame['optimization_eligible_date'].isna()
        | (pd.to_datetime(feature_frame['optimization_eligible_date']) <= pd.Timestamp(SNAPSHOT))
    )
)

position_bins = [0, 5, 10, 20]
position_labels = ['1-5', '6-10', '11-20']
feature_frame['position_band'] = pd.cut(
    feature_frame['avg_position_mar'].where(
        (feature_frame['avg_position_mar'] > 0) & (feature_frame['avg_position_mar'] <= 20)
    ),
    bins=position_bins, labels=position_labels,
)

eligible_page12_mask = feature_frame['is_eligible'] & feature_frame['position_band'].notna()
peer_ctr_full = (
    feature_frame.loc[eligible_page12_mask]
    .groupby('position_band', observed=True)['ctr_mar']
    .median()
)
feature_frame['peer_median_ctr'] = feature_frame['position_band'].map(peer_ctr_full).astype(float)

feature_frame['is_stale'] = feature_frame['days_since_update'] >= 180
feature_frame['ctr_position_gap'] = (
    feature_frame['position_band'].notna()
    & (feature_frame['ctr_mar'] < 0.5 * feature_frame['peer_median_ctr'])
)

feature_frame['baseline_score'] = (
    feature_frame['is_eligible'].astype(int)
    * feature_frame['is_stale'].astype(int)
    * feature_frame['ctr_position_gap'].astype(int)
    * feature_frame['gsc_impressions_mar']
)
feature_frame['baseline_action_label'] = np.where(
    feature_frame['baseline_score'] > 0, 'review_for_refresh', 'no_action_flagged'
)

n_flagged = (feature_frame['baseline_score'] > 0).sum()
print(f'Week-4 baseline, rebuilt here: {n_flagged} of {len(feature_frame)} pages flagged '
      f'({n_flagged / len(feature_frame):.1%}) -- this should match what w04_baseline_score.ipynb '
      f'wrote to work/outputs/baseline_action_score.csv. If it does not, stop and diff the two builds '
      f'before trusting anything below.')

Week-4 baseline, rebuilt here: 1 of 134086 pages flagged (0.0%) -- this should match what w04_baseline_score.ipynb wrote to work/outputs/baseline_action_score.csv. If it does not, stop and diff the two builds before trusting anything below.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


w02 already tested the tempting shortcut — a single-signal threshold rule — and killed it: every
candidate feature correlated weakly with the label tested there (strongest was `content_age_days`
at -0.164, though that test ran on the starter CSV's `is_declining_label`, not the warehouse's
`is_declining_proxy` used from w03 onward — the same weak-signal conclusion, but not yet re-tested
on the actual label this model predicts). w04's baseline confirmed the practical consequence of
that weak-signal world, and it's worse than "just low recall": the AND-gate rule
(`eligible × stale × ctr_gap`) only fires when *every* condition lines up, which flags exactly 1 of
134,086 pages (see 0.2 above) — and that one deliberate pick is a **false positive**
(`is_declining_proxy = 0`). Every other row in the ranked list is tied at a score of 0, so the
reported precision@10/20/50/100 numbers mostly reflect how ties happen to sort, not a real
ranking — the only non-arbitrary output the rule produces is wrong. A rule whose single confident
pick misses isn't a usable prioritization tool.

That's exactly the situation the toolkit's tree-based methods are built for: signal "spread
thinly across several weak, interacting signals at once" (w02's own words) is precisely what a
Random Forest can learn and a hand-written AND-gate can't. So I'm training two models, not one:

- **Logistic Regression** — a linear, coefficient-interpretable model. It's the honest control: if
  a linear combination of the same five to nine features already beats the baseline, I don't need
  a black box to explain why.
- **Random Forest** — the model built to catch weak, interacting signals a linear model and a
  hand-written rule both miss. Depth and leaf size are kept shallow (`max_depth=8`,
  `min_samples_leaf=20`) on purpose: with correlations this weak, an unconstrained forest would
  just memorize per-client noise, not learn a real pattern (w04's assignment note: "does not
  reward complexity alone").

I'm not running Gradient Boosting or clustering here — the lane's decision (rank pages for review)
is a straightforward binary-classification-into-ranking problem (w02 Section 1), and two models
that both beat or fail to beat the same baseline on the same split already answers the question
honestly without adding a third result to explain away.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [27]:
from sklearn.model_selection import GroupKFold

groups = feature_frame['client_hash_id'].values
gkf = GroupKFold(n_splits=5)

n_clients = feature_frame['client_hash_id'].nunique()
print(f'{n_clients} distinct clients across {len(feature_frame)} pages -- GroupKFold(5) keeps every ')
print('client entirely inside one fold, so no page from a test-fold client was ever seen in training.')

41 distinct clients across 134086 pages -- GroupKFold(5) keeps every 
client entirely inside one fold, so no page from a test-fold client was ever seen in training.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### 3.1 Feature matrix

Same data-contract rules as w03 Section 2 apply: only fields knowable at the March 31 decision moment go in. `gsc_impressions_feb`, `pct_change_mar_vs_feb`, and `is_declining_proxy` itself are excluded on purpose (the leakage trap w03 built and tested). Two additions beyond the baseline's own feature set, both still legal by the same contract: `word_count`/`backlinks`/`content_type` (already whitelisted as features in w03 Section 2, just unused by the simpler rule) and `ga4_engagement_rate_mar` (present in w03's original 5-feature set, dropped from w04's leaner baseline build but restored here). GA4 coverage is partial (w03 found it non-null for ~38% of rows) — per the `flyrank-data` skill's warning against a blind `fillna(0)` injecting a false signal, missingness gets its own explicit flag column rather than being silently zero-filled.

In [28]:
# GA4 engagement rate -- explicit availability flag, not a blind fillna (flyrank-data skill warning).
feature_frame['has_ga4_mar'] = (
    feature_frame['ga4_pageviews_mar'].notna() & (feature_frame['ga4_pageviews_mar'] > 0)
)
feature_frame['ga4_engagement_rate_mar'] = np.where(
    feature_frame['has_ga4_mar'],
    feature_frame['ga4_engaged_sessions_mar'] / feature_frame['ga4_pageviews_mar'].replace(0, np.nan),
    0.0,
)
feature_frame['ga4_engagement_rate_mar'] = feature_frame['ga4_engagement_rate_mar'].fillna(0.0)

# word_count -- missingness follows content_type per the flyrank-data skill; flag it, don't hide it.
feature_frame['has_word_count'] = feature_frame['word_count'].notna()
feature_frame['word_count_filled'] = feature_frame['word_count'].fillna(feature_frame['word_count'].median())
feature_frame['backlinks_filled'] = feature_frame['backlinks'].fillna(0)

# Impressions are heavily right-skewed (w03: mean ~1,948, max 617,124) -- log1p for the linear model's benefit;
# harmless for the tree model, which is scale-invariant.
feature_frame['log_impressions_mar'] = np.log1p(feature_frame['gsc_impressions_mar'])

content_type_dummies = pd.get_dummies(feature_frame['content_type'], prefix='ctype', dummy_na=True)

NUMERIC_FEATURES = [
    'log_impressions_mar', 'ctr_mar', 'avg_position_mar', 'content_age_days',
    'days_since_update', 'word_count_filled', 'backlinks_filled',
    'ga4_engagement_rate_mar', 'has_ga4_mar', 'has_word_count',
]

X = pd.concat(
    [feature_frame[NUMERIC_FEATURES].astype(float), content_type_dummies.astype(float)], axis=1
).fillna(0.0)
y = feature_frame['is_declining_proxy'].values

print(f'Model matrix: {X.shape[0]} rows x {X.shape[1]} columns')
print('Feature columns:', list(X.columns))

Model matrix: 134086 rows x 14 columns
Feature columns: ['log_impressions_mar', 'ctr_mar', 'avg_position_mar', 'content_age_days', 'days_since_update', 'word_count_filled', 'backlinks_filled', 'ga4_engagement_rate_mar', 'has_ga4_mar', 'has_word_count', 'ctype_comparison article', 'ctype_feedly article', 'ctype_keyword article', 'ctype_nan']


In [29]:
# Leakage check, same style as w03 Section 3.6 / w04's Section 4 assert -- run it BEFORE training, not after.
forbidden = {'is_declining_proxy', 'gsc_impressions_feb', 'pct_change_mar_vs_feb',
             'baseline_score', 'baseline_action_label'}
leak = forbidden & set(X.columns)
assert not leak, f'Leaked columns made it into the model matrix: {leak}'
print('OK: no label-derived or baseline-derived columns are in the model feature matrix.')
print(f'OK: feature window is {MONTH} only -- no report_date beyond it was ever queried.')

OK: no label-derived or baseline-derived columns are in the model feature matrix.
OK: feature window is 2026-03 only -- no report_date beyond it was ever queried.


### 3.2 Train both models with out-of-fold, client-grouped predictions

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

logreg = Pipeline([
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
])

rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1,
)

oof_logreg = cross_val_predict(logreg, X, y, cv=gkf, groups=groups, method='predict_proba')[:, 1]
oof_rf = cross_val_predict(rf, X, y, cv=gkf, groups=groups, method='predict_proba')[:, 1]

feature_frame['pred_logreg'] = oof_logreg
feature_frame['pred_rf'] = oof_rf

print('Out-of-fold probabilities computed for both models -- 5 folds, grouped by client_hash_id.')
print('Every prediction above comes from a fold where that row\'s client was held out of training.')

Out-of-fold probabilities computed for both models -- 5 folds, grouped by client_hash_id.
Every prediction above comes from a fold where that row's client was held out of training.


### 3.3 AUC — model vs. baseline rule, same split, same label

In [31]:
from sklearn.metrics import roc_auc_score

# AUC is threshold-free and rank-based, so it's a fair way to score the baseline's 0/score column too --
# even though almost all of that column is tied at 0.
auc_baseline = roc_auc_score(y, feature_frame['baseline_score'])
auc_logreg = roc_auc_score(y, oof_logreg)
auc_rf = roc_auc_score(y, oof_rf)

print(f'AUC -- baseline rule score  : {auc_baseline:.3f}')
print(f'AUC -- logistic regression  : {auc_logreg:.3f}')
print(f'AUC -- random forest        : {auc_rf:.3f}')

best_name, best_auc = max(
    [('logistic regression', auc_logreg), ('random forest', auc_rf)], key=lambda t: t[1]
)
lift = best_auc - auc_baseline
print()
if lift > 0.05:
    print(f'VERDICT: {best_name} beats the baseline rule score by {lift:+.3f} AUC -- a real, '
          f'non-trivial improvement worth reporting as the headline result.')
elif lift > 0.01:
    print(f'VERDICT: {best_name} edges out the baseline rule score by {lift:+.3f} AUC -- a real but '
          f'modest improvement; report it plainly, do not oversell it.')
elif lift > -0.01:
    print(f'VERDICT: {best_name} and the baseline rule score are essentially tied ({lift:+.3f} AUC) -- '
          f'the honest conclusion is the transparent rule already captures most of the separable signal '
          f'here, and the model\'s main advantage (if any) is a usable continuous ranking, not raw AUC.')
else:
    print(f'VERDICT: neither model beats the baseline rule score on AUC ({lift:+.3f}) -- keep the '
          f'baseline as the reported number and treat the model as exploratory, not a replacement.')

AUC -- baseline rule score  : 0.500
AUC -- logistic regression  : 0.719
AUC -- random forest        : 0.723

VERDICT: random forest beats the baseline rule score by +0.223 AUC -- a real, non-trivial improvement worth reporting as the headline result.


### 3.4 precision@K — the number an editor actually feels

Same metric, same K values, same population as w04 Section 2.1, so this table sits directly next to that notebook's result.

In [32]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = feature_frame['is_declining_proxy'].mean()
print(f'Base rate (share of ALL {len(feature_frame)} pages that are is_declining_proxy=1): {base_rate:.3f}\n')

rows = []
for k in (10, 20, 50, 100):
    rows.append({
        'K': k,
        'baseline_precision': precision_at_k(feature_frame['baseline_score'], y, k),
        'logreg_precision': precision_at_k(oof_logreg, y, k),
        'rf_precision': precision_at_k(oof_rf, y, k),
    })
comparison = pd.DataFrame(rows).set_index('K')
comparison['baseline_lift'] = comparison['baseline_precision'] - base_rate
comparison['logreg_lift'] = comparison['logreg_precision'] - base_rate
comparison['rf_lift'] = comparison['rf_precision'] - base_rate
comparison.round(3)

Base rate (share of ALL 134086 pages that are is_declining_proxy=1): 0.201



,baseline_precision,logreg_precision,rf_precision,baseline_lift,logreg_lift,rf_lift
K,,,,,,
10,0.40,0.80,1.00,0.199,0.599,0.799
20,0.35,0.80,1.00,0.149,0.599,0.799
50,0.22,0.62,0.96,0.019,0.419,0.759
100,0.14,0.69,0.95,-0.061,0.489,0.749


In [33]:
n_baseline_nonzero = (feature_frame['baseline_score'] > 0).sum()
print(f'Context for reading the table above: only {n_baseline_nonzero} of {len(feature_frame)} rows have a '
      f'nonzero baseline_score. That means "precision@10/20/50/100" for the baseline column is really '
      f'precision over an arbitrary tie-break order among score=0 rows for all K past {n_baseline_nonzero} -- '
      f'not a meaningful ranking beyond its first {n_baseline_nonzero} row(s). The model columns are '
      f'continuous probabilities with genuine spread across all {len(feature_frame)} rows, so their '
      f'precision@K numbers reflect real ranking quality at every K, not tie-break luck. This is the '
      f'model\'s practical advantage even in scenarios where the AUC gap above is small: it produces a '
      f'full, orderable review queue, not a single flagged page.')

Context for reading the table above: only 1 of 134086 rows have a nonzero baseline_score. That means "precision@10/20/50/100" for the baseline column is really precision over an arbitrary tie-break order among score=0 rows for all K past 1 -- not a meaningful ranking beyond its first 1 row(s). The model columns are continuous probabilities with genuine spread across all 134086 rows, so their precision@K numbers reflect real ranking quality at every K, not tie-break luck. This is the model's practical advantage even in scenarios where the AUC gap above is small: it produces a full, orderable review queue, not a single flagged page.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### 4.1 Permutation importance — what is the Random Forest actually leaning on?

Fit once on a single grouped train/test split (permutation importance needs one fitted model, not a cross-validated ensemble of five) purely to inspect feature weight — the AUC/precision numbers above, from the full 5-fold out-of-fold run, remain the reported metrics.

In [34]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.inspection import permutation_importance

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

rf_fitted = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1,
).fit(X.iloc[train_idx], y[train_idx])

perm = permutation_importance(
    rf_fitted, X.iloc[test_idx], y[test_idx],
    n_repeats=10, random_state=42, scoring='roc_auc', n_jobs=-1,
)
importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

print('Permutation importance (drop in held-out AUC when a feature is shuffled -- bigger = more load-bearing):')
print(importance.round(4).head(10))

top_feature = importance.index[0]
print(f'\nThe single most load-bearing feature on this held-out client split is `{top_feature}`.')
print('Compare this against w02\'s single-signal correlation table (content_age_days led at only -0.164) --')
print('if a different feature or combination leads here, that itself is evidence the model is using')
print('interaction structure a single-column correlation could never surface.')

Permutation importance (drop in held-out AUC when a feature is shuffled -- bigger = more load-bearing):
log_impressions_mar      0.1317
avg_position_mar         0.0194
content_age_days         0.0086
has_ga4_mar              0.0035
days_since_update        0.0033
ctr_mar                  0.0026
ctype_feedly article     0.0024
has_word_count           0.0016
word_count_filled        0.0004
ctype_keyword article    0.0003
dtype: float64

The single most load-bearing feature on this held-out client split is `log_impressions_mar`.
Compare this against w02's single-signal correlation table (content_age_days led at only -0.164) --
if a different feature or combination leads here, that itself is evidence the model is using
interaction structure a single-column correlation could never surface.


### 4.2 False positives — where the model is confidently wrong

In [35]:
review = feature_frame.copy()

top20_model = review.sort_values('pred_rf', ascending=False).head(20)
false_positives = top20_model[top20_model['is_declining_proxy'] == 0]

print(f'Of the model\'s top 20 highest-probability pages, {len(false_positives)} are NOT actually declining '
      f'by the proxy label -- these are the false positives worth a skeptical look:\n')
false_positives[['content_hash_id', 'pred_rf', 'gsc_impressions_mar', 'ctr_mar',
                  'avg_position_mar', 'days_since_update', 'content_age_days']].round(3)

Of the model's top 20 highest-probability pages, 0 are NOT actually declining by the proxy label -- these are the false positives worth a skeptical look:



,content_hash_id,pred_rf,gsc_impressions_mar,ctr_mar,avg_position_mar,days_since_update,content_age_days


### 4.3 False negatives — declining pages the model was least confident about

In [36]:
declining = review[review['is_declining_proxy'] == 1].sort_values('pred_rf', ascending=True)
false_negatives = declining.head(10)

print('10 actually-declining pages the model gave the LOWEST predicted probability -- these are the misses:\n')
false_negatives[['content_hash_id', 'pred_rf', 'gsc_impressions_mar', 'ctr_mar',
                  'avg_position_mar', 'days_since_update', 'content_age_days']].round(3)

10 actually-declining pages the model gave the LOWEST predicted probability -- these are the misses:



,content_hash_id,pred_rf,gsc_impressions_mar,ctr_mar,avg_position_mar,days_since_update,content_age_days
88131,content_b0e4211ec207c90d,0.073,8526.0,0.001,38.793,34,83
115098,content_d2687681261f4a4a,0.077,8831.0,0.002,34.540,0,146
116009,content_e7842b8465823f63,0.078,8510.0,0.000,37.554,34,83
88333,content_b511951068f01fca,0.084,6815.0,0.002,32.248,0,141
82380,content_3282339607cc0ac4,0.086,5351.0,0.005,14.190,0,69
85811,content_7de4f81725f32ff9,0.087,8685.0,0.000,37.724,0,159
111706,content_3ee8ba72305fcbfb,0.089,25208.0,0.008,7.927,0,77
81864,content_2670233a3c0d6fc6,0.090,4498.0,0.006,16.584,0,83
10853,content_096dc0608871836f,0.090,9882.0,0.002,5.617,0,78
82232,content_2f07c93433632aa7,0.091,9196.0,0.003,29.287,0,146


### 4.4 Reading the errors together

**Top 20 by predicted probability: zero false positives.** On this out-of-fold run, every one of the model's
20 highest-confidence pages is actually declining by the proxy label (`false_positives` returned 0 rows —
precision@20 = 1.0). That's a strong result, but it's a result about the *very top* of the queue only — it says
nothing about how the model behaves at K=50 or K=100 (see Section 3.4 for that), and one clean top-20 on one
split isn't proof the model never produces a false positive, just that it didn't here.

**The model is leaning almost entirely on one signal.** Permutation importance (4.1) puts `log_impressions_mar`
at 0.1317 — nearly 7x the next feature (`avg_position_mar` at 0.0194), and over an order of magnitude ahead of
`content_age_days`, `days_since_update`, and `ctr_mar`. That's worth stating plainly rather than letting the
"interacting weak signals" framing from Section 1 stand unchallenged: this Random Forest's real skill is
concentrated in *how much current traffic a page has*, not in the freshness/staleness combination the baseline
rule was built around. That's plausibly a legitimate signal — a bigger, more established page is more
traffic-stable, so the same percentage drop is rarer and more meaningful when it happens on one of those pages —
but it's also a narrower story than "the model learned the interaction the rule couldn't," and the honest
version of this section says so.

**The false negatives are not what I expected going in.** The hypothesis in the original draft of this section
was that misses would be strong pages that fell from an unusually high February — a seasonality artifact in the
proxy label. The actual top-10 misses (4.3) don't support that: `days_since_update` is 0 for 8 of the 10, and 34
for the other two — every single one of these pages was updated at most a month before the snapshot, nowhere
close to the baseline's 180-day staleness gate. These are pages that were *just refreshed* and are declining
anyway. That's a more concrete and more useful finding than the generic seasonality story: it's a case the
baseline rule was structurally incapable of ever flagging (its `is_stale` gate requires `days_since_update >=
180`), and it suggests the model's blind spot here is recently-touched pages whose refresh didn't work — not
proxy-label noise.

**Compare to the baseline's own blind spot.** The baseline flagged at most a handful of pages total (Section
0.2) — it has *no* false positives among its flagged set almost by construction (it barely flags anything), but
that same conservatism means it has enormous numbers of unexamined false negatives: any genuinely declining page
that fails even one of the three AND-gates (recently updated, as the 4.3 pages above show, or just under the
impressions/age threshold) gets zero score and never enters the conversation. The model's errors are visible and
countable; the baseline's errors are mostly invisible, hidden inside `no_action_flagged`. That asymmetry is
itself a finding.

**Interpretability is not free with the model.** The baseline's `reason_code` is a sentence a non-engineer can
read ("stale and underperforming its peers"). The model's `pred_rf` is a probability driven almost entirely by
one feature (impression volume) plus a long tail of much weaker contributors — not a per-row reason, and not the
multi-signal story Section 1 originally hoped for. A reasonable production compromise: use the model's score to
build the ranked queue and report the AUC/precision numbers, but keep the baseline's `reason_code` attached to
any row where it also fires, so the pages an editor is most likely to trust first come with a plain-English
reason — and flag recently-updated-but-still-declining pages (the 4.3 pattern) as a distinct, model-only category
the baseline will never surface on its own.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.